In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = digits.images.reshape(len(digits.images), -1)
y = digits.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
train_mean = np.mean(X_train, axis=0)
train_std = np.std(X_train, axis=0)
train_std[train_std == 0] = 1
X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

In [ ]:
print(train_std)
print(train_mean)

[1.         0.90831445 4.80739606 4.28461944 4.277748   5.63169185
 3.39181301 1.10477323 0.0912126  3.2130994  5.45332348 3.99562405
 4.75620514 6.11280933 3.62500833 0.87665107 0.06970762 3.54122308
 5.70994107 5.85316015 6.15088288 6.21619878 3.28717156 0.43305649
 0.03728071 3.06026936 6.18655946 5.92608632 6.16300002 5.88463035
 3.72004641 0.04564345 1.         3.43447566 6.30103834 6.28587936
 5.92277974 5.85315718 3.52022356 1.         0.16012222 2.96659879
 6.52785468 6.41339024 6.22789376 5.71343338 4.3379782  0.34041962
 0.22827664 1.7794463  5.64537686 5.24067799 5.24602804 6.00797456
 4.9026196  0.93715949 0.02637063 0.91870531 5.13122333 4.39398513
 4.8898812  5.89198003 4.14184433 1.87478321]
[0.00000000e+00 3.10368824e-01 5.22755741e+00 1.18218511e+01
 1.18907446e+01 5.84133612e+00 1.37787056e+00 1.44745999e-01
 5.56715379e-03 1.98329854e+00 1.03159360e+01 1.19860821e+01
 1.03757829e+01 8.28949200e+00 1.90187891e+00 1.16910230e-01
 3.47947112e-03 2.54488518e+00 9.7842727

In [ ]:
def get_euclidean(train_data, test_point):
    return np.sqrt(np.sum((train_data - test_point)**2, axis=1))

def get_manhattan(train_data, test_point):
    return np.sum(np.abs(train_data - test_point), axis=1)

def get_mahalanobis(train_data, test_point, inv_cov):
    diff = train_data - test_point
    dist = np.sqrt(np.sum(np.dot(diff, inv_cov) * diff, axis=1))
    return dist

In [ ]:
def predict_knn(X_train, y_train, X_test, k=5, metric='euclidean', inv_cov=None):
    predictions = []

    for test_point in X_test:
        if metric == 'euclidean':
            distances = get_euclidean(X_train, test_point)
        elif metric == 'manhattan':
            distances = get_manhattan(X_train, test_point)
        elif metric == 'mahalanobis':
            distances = get_mahalanobis(X_train, test_point, inv_cov)

        k_indices = np.argsort(distances)[:k]
        k_nearest_labels = y_train[k_indices]
        counts = np.bincount(k_nearest_labels)
        predictions.append(np.argmax(counts))

    return np.array(predictions)

cov_matrix = np.cov(X_train.T)
inv_cov = np.linalg.pinv(cov_matrix)

metrics = ['euclidean', 'manhattan', 'mahalanobis']

print(f"{'Metric':<15} | {'Accuracy':<10}")
print("-" * 30)

for m in metrics:
    y_pred = predict_knn(X_train, y_train, X_test[:200], k=5, metric=m, inv_cov=inv_cov)
    accuracy = np.mean(y_pred == y_test[:200])
    print(f"{m:<15} | {accuracy * 100:.2f}%")

Metric          | Accuracy  
------------------------------
euclidean       | 96.50%
manhattan       | 98.00%
mahalanobis     | 93.00%
